In [1]:
import torch

In [ ]:
'''
learning how to build the pytorchified training framework
'''

In [33]:
class Layer:
    def __init__(self):
        self.w = 2
        self.b = 3
        self.parameters = [self.w, self.b]

    def __call__(self, x):
        out = x * self.w + self.b
        return out

class Model:
    def __init__(self, list_layers):
        self.layers = list_layers
        self.parameters = []
        for layer in list_layers:
            for layer_parameters in layer.parameters:
                self.parameters.append(layer_parameters)# But what if a layer doesn't have a parameter?

new_model = Model([Layer(), Layer(), layer()])

for Layer in new_model.layers:
    print(Layer.parameters)
    print(Layer(1))

print(new_model.parameters)

[2, 3]
5
[2, 3]
5
[2, 3]
5
[2, 3, 2, 3, 2, 3]


In [177]:
class Parameter:

    def __init__(self, tensor: torch.Tensor):

        if not isinstance(tensor, torch.Tensor):
            raise TypeError(f"Expected tensor, got {type(tensor)}")
        self.data = tensor

class Module:

    def parameters(self): # Note this is kind of scrappy and lacks robustness, but it does the job for now helping to conceptualize a general pytorch-like API
        params = []
        for key, value in self.__dict__.items():
            if isinstance(value, Parameter):
                params.append(value.data)
            elif isinstance(value, Module):
                for param in value.parameters():
                    params.append(param)
            elif isinstance(value, list):
                for module in value:
                    for param in module.parameters():
                        params.append(param)
        return params

class Sequential(Module):
    
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, x):
        x
        for layer in self.layers:
            x = layer(x)
        return x    

class Tanh(Module):
    
    def __call__(self, x):
        out = torch.tanh(x)
        return out

class Linear(Module):

    def __init__(self, in_features, out_features):
        self.w = Parameter(torch.randn(in_features, out_features))
        self.b = Parameter(torch.randn(out_features))
        
    def __call__(self, x):
        out = x @ self.w.data + self.b.data
        return out

model = Sequential([
    Linear(10, 5),
    Tanh(),
])

print(f"Passing x = 10 through model: {model(torch.randn(10))}\n")
print(f"model's parameters: {model.parameters()}")

Passing x = 10 through model: tensor([-1.0000, -0.5625, -0.8627, -0.1568, -1.0000])

model's parameters: [tensor([[-0.2654, -2.2949, -2.7560,  0.2126, -1.3420],
        [ 1.9552,  0.3812, -0.6214,  1.8237,  0.5465],
        [ 0.5919,  0.7913,  1.2170,  0.4536, -0.3975],
        [ 0.1091, -0.7485,  0.6039, -0.4704,  1.0834],
        [-1.6513, -0.4732, -2.5401,  1.9139,  0.7750],
        [ 0.5305,  1.4230,  0.1528,  0.5577, -1.7347],
        [ 0.5530, -0.4750, -0.1919, -1.0249,  1.0003],
        [-0.5830, -1.2430,  0.0835, -0.3362,  0.3372],
        [-0.7483,  1.8826,  2.1292, -0.8297, -0.4384],
        [-1.2863, -1.6712,  1.3340, -0.7273, -0.8454]]), tensor([-0.7770,  1.9833, -0.2164,  0.0400, -0.1338])]
